# Persian G2P VITS training

This notebook trains a Persian VITS model whose input is explicitly produced by Homo-GE2PE. The pipeline is:

`Persian text -> Homo-GE2PE -> phoneme sequence -> Coqui VITS -> waveform`

VITS never sees the original Persian graphemes. The same G2P stage is therefore required at inference time.

## 1. Install the project

Use Python 3.10–3.13. A CUDA GPU is strongly recommended for training.

In [ ]:
from pathlib import Path
import os
import subprocess

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from the repository.'
os.chdir(ROOT)
PROJECT_PYTHON = ROOT / '.venv/bin/python'
if not PROJECT_PYTHON.exists():
    subprocess.run(['bash', 'scripts/setup.sh'], check=True)
# All model work below uses this interpreter, even if the notebook itself
# is opened with a different Jupyter kernel.
subprocess.run([str(PROJECT_PYTHON), '-m', 'pip', 'install', '--no-deps', '-e', '.'], check=True)
print('Project root:', ROOT)
print('Model Python:', PROJECT_PYTHON)

In [ ]:
from pathlib import Path
import os
import subprocess

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
PROJECT_PYTHON = ROOT / '.venv/bin/python'
assert PROJECT_PYTHON.exists(), 'Run the setup cell once to create .venv.'

probe = subprocess.run(
    [str(PROJECT_PYTHON), '-c', 'import torch; print("PyTorch:", torch.__version__); print("CUDA available:", torch.cuda.is_available()); print("MPS available:", torch.backends.mps.is_available())'],
    check=True, text=True, capture_output=True,
)
print(probe.stdout)

## 2. Verify the Persian G2P stage

The first run downloads the Homo-GE2PE checkpoint.

In [ ]:
from pathlib import Path
import os
import subprocess

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
PROJECT_PYTHON = ROOT / '.venv/bin/python'
assert PROJECT_PYTHON.exists(), 'Run the setup cell once to create .venv.'

example_text = 'سلام، این یک آزمایش تبدیل متن فارسی به گفتار است.'
result = subprocess.run(
    [str(PROJECT_PYTHON), 'scripts/phonemize.py', '--text', example_text],
    check=True, text=True, capture_output=True,
)
example_phonemes = result.stdout.strip()
print('Text:', example_text)
print('Phonemes:', example_phonemes)

## 3. Prepare ManaTTS

The default experiment uses 1,000 utterances from the first ManaTTS shard. Increase `MAX_SAMPLES` for a serious training run. Existing prepared data is preserved unless `--overwrite` is included.

In [ ]:
from pathlib import Path
import os
import subprocess

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
PROJECT_PYTHON = ROOT / '.venv/bin/python'
assert PROJECT_PYTHON.exists(), 'Run the setup cell once to create .venv.'

MAX_SAMPLES = 1000
prepare_command = [
    str(PROJECT_PYTHON), 'scripts/prepare_dataset.py',
    '--max-samples', str(MAX_SAMPLES),
    '--batch-size', '16',
]
subprocess.run(prepare_command, check=True)

In [ ]:
from pathlib import Path
import os
import json

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

audit_path = ROOT / 'data/manatts-g2p/metadata.jsonl'
with audit_path.open(encoding='utf-8') as stream:
    for _ in range(3):
        row = json.loads(next(stream))
        print(row['text'])
        print(' ->', row['phonemes'], '\n')

## 4. Train VITS

A short run only checks that the pipeline works. High-quality synthesis requires substantially more data and training time. Checkpoints and TensorBoard logs are saved in `outputs/`.

In [ ]:
from pathlib import Path
import os
import subprocess

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
PROJECT_PYTHON = ROOT / '.venv/bin/python'
assert PROJECT_PYTHON.exists(), 'Run the setup cell once to create .venv.'

EPOCHS = 1000
BATCH_SIZE = 16
train_command = [
    str(PROJECT_PYTHON), 'scripts/train_vits.py',
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
]
subprocess.run(train_command, check=True)

## 5. Synthesize with G2P + VITS

This cell locates the newest checkpoint, converts Persian text to phonemes, and writes a WAV file.

In [ ]:
from pathlib import Path
import os
import subprocess

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
PROJECT_PYTHON = ROOT / '.venv/bin/python'
assert PROJECT_PYTHON.exists(), 'Run the setup cell once to create .venv.'

output_wav = ROOT / 'outputs/notebook_sample.wav'
subprocess.run([
    str(PROJECT_PYTHON), 'scripts/infer.py',
    '--text', 'امروز هوا بسیار خوب است.',
    '--output', str(output_wav),
], check=True)
from IPython.display import Audio
Audio(filename=str(output_wav))